# CIFAKE EDA

이 노트북에서는 CIFAKE 데이터셋의 구조와 품질을 점검하고,
REAL / FAKE 이미지 간 차이를 시각적/통계적/주파수적 관점에서 탐색한다.

## 목표
1. 데이터셋 구조와 무결성 확인
2. REAL / FAKE의 기본 시각적 차이 확인
3. 픽셀 통계량 및 텍스처 차이 비교
4. 주파수 도메인 차이 확인
5. 임베딩 공간에서의 분포 차이 확인
6. 중복/누수 가능성 탐색

## 기대 산출물
- 데이터셋 sanity check 결과
- REAL/FAKE 분포 비교 그래프
- 주파수 분석 결과
- 모델링 방향에 대한 인사이트

## 1. 환경 설정

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image, ImageOps
import cv2

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_ROOT = Path("./data/raw")

# 필요에 따라 실제 경로 확인 후 수정
list(DATA_ROOT.iterdir())[:10]

> 먼저 실제 Kaggle input 구조를 확인한 뒤, train/test/REAL/FAKE 경로를 지정한다.

In [ ]:
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

TRAIN_REAL_DIR = TRAIN_DIR / "REAL"
TRAIN_FAKE_DIR = TRAIN_DIR / "FAKE"
TEST_REAL_DIR = TEST_DIR / "REAL"
TEST_FAKE_DIR = TEST_DIR / "FAKE"

for p in [TRAIN_REAL_DIR, TRAIN_FAKE_DIR, TEST_REAL_DIR, TEST_FAKE_DIR]:
    print(p, p.exists())

## 2. 헬퍼 함수 정의

In [ ]:
def get_image_paths(root_dir, split, label):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths = []
    for p in sorted(root_dir.rglob("*")):
        if p.suffix.lower() in exts:
            paths.append({
                "filepath": str(p),
                "filename": p.name,
                "split": split,
                "label": label
            })
    return paths

In [ ]:
def image_basic_stats(img):
    arr = np.array(img).astype(np.float32) / 255.0
    
    gray = cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    
    brightness = gray.mean()
    contrast = gray.std()
    rgb_mean = arr.mean(axis=(0, 1))
    rgb_std = arr.std(axis=(0, 1))
    
    # Shannon entropy
    hist, _ = np.histogram(gray.flatten(), bins=256, range=(0, 1), density=True)
    hist = hist + 1e-12
    entropy = -(hist * np.log2(hist)).sum()
    
    # Sharpness (Laplacian variance)
    lap_var = cv2.Laplacian((gray * 255).astype(np.uint8), cv2.CV_64F).var()
    
    # Edge density
    edges = cv2.Canny((gray * 255).astype(np.uint8), 100, 200)
    edge_density = (edges > 0).mean()
    
    return {
        "width": img.size[0],
        "height": img.size[1],
        "brightness": brightness,
        "contrast": contrast,
        "r_mean": rgb_mean[0],
        "g_mean": rgb_mean[1],
        "b_mean": rgb_mean[2],
        "r_std": rgb_std[0],
        "g_std": rgb_std[1],
        "b_std": rgb_std[2],
        "entropy": entropy,
        "lap_var": lap_var,
        "edge_density": edge_density
    }

In [ ]:
def compute_fft_features(img):
    gray = np.array(img.convert("L")).astype(np.float32) / 255.0
    
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    magnitude = np.log1p(np.abs(fshift))
    
    h, w = magnitude.shape
    cy, cx = h // 2, w // 2
    y, x = np.indices((h, w))
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    
    r = r.astype(np.int32)
    radial_mean = np.bincount(r.ravel(), magnitude.ravel()) / np.bincount(r.ravel())
    
    # 중심부(저주파) / 바깥쪽(고주파) 에너지 비율
    max_r = r.max()
    low_mask = r <= max_r * 0.25
    high_mask = r >= max_r * 0.60
    
    low_energy = magnitude[low_mask].mean()
    high_energy = magnitude[high_mask].mean()
    
    return {
        "fft_magnitude": magnitude,
        "radial_profile": radial_mean,
        "low_freq_energy": low_energy,
        "high_freq_energy": high_energy,
        "high_low_ratio": high_energy / (low_energy + 1e-8)
    }

In [ ]:
def show_image_grid(df, n=16, title="Sample Images"):
    sample_df = df.sample(min(n, len(df)), random_state=SEED).reset_index(drop=True)
    
    cols = 4
    rows = int(np.ceil(len(sample_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = np.array(axes).reshape(-1)
    
    for ax in axes:
        ax.axis("off")
    
    for i, row in sample_df.iterrows():
        img = Image.open(row["filepath"]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f'{row["split"]} | {row["label"]}', fontsize=10)
        axes[i].axis("off")
    
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

## 3. Build Metadata Table

이미지 파일 경로와 기본 메타데이터를 수집해 하나의 테이블로 정리한다.

In [ ]:
records = []
records += get_image_paths(TRAIN_REAL_DIR, "train", "REAL")
records += get_image_paths(TRAIN_FAKE_DIR, "train", "FAKE")
records += get_image_paths(TEST_REAL_DIR, "test", "REAL")
records += get_image_paths(TEST_FAKE_DIR, "test", "FAKE")

df = pd.DataFrame(records)
df.shape

In [ ]:
df.head()

In [ ]:
df.groupby(["split", "label"]).size().unstack(fill_value=0)

In [ ]:
df["filesize"] = df["filepath"].apply(lambda x: os.path.getsize(x))
df["filesize_kb"] = df["filesize"] / 1024
df["ext"] = df["filepath"].apply(lambda x: Path(x).suffix.lower())

df.head()

## 4. Dataset Integrity Check

- 손상 파일이 있는지
- 이미지 크기와 채널 수가 일관적인지
- split / label 분포가 정상인지 확인한다.

In [ ]:
stats_list = []
errors = []

for path in tqdm(df["filepath"], total=len(df)):
    img, err = safe_open_image(path)
    if err is not None:
        errors.append((path, err))
        stats_list.append({
            "filepath": path,
            "is_corrupted": True
        })
    else:
        s = image_basic_stats(img)
        s["filepath"] = path
        s["is_corrupted"] = False
        stats_list.append(s)

stats_df = pd.DataFrame(stats_list)
df = df.merge(stats_df, on="filepath", how="left")

In [ ]:
print("Corrupted files:", df["is_corrupted"].sum())
errors[:5]

In [ ]:
df[["width", "height"]].value_counts().head(10)

In [ ]:
df.groupby(["split", "label"])[["width", "height"]].agg(["nunique", "mean"]).round(2)

In [ ]:
df.groupby(["split", "label"])[["width", "height"]].agg(["nunique", "mean"]).round(2)

여기서 확인할 포인트:
- 모든 이미지가 동일 해상도인지
- corrupted file이 없는지
- split/label 분포가 기대와 맞는지

## 5. Quick Visual Inspection

REAL / FAKE 이미지를 직접 비교해 전반적인 질감, 경계, 색감 차이를 먼저 살펴본다.

In [ ]:
show_image_grid(df[(df["split"] == "train") & (df["label"] == "REAL")], n=16, title="Train REAL Samples")
show_image_grid(df[(df["split"] == "train") & (df["label"] == "FAKE")], n=16, title="Train FAKE Samples")
show_image_grid(df[(df["split"] == "test") & (df["label"] == "REAL")], n=16, title="Test REAL Samples")
show_image_grid(df[(df["split"] == "test") & (df["label"] == "FAKE")], n=16, title="Test FAKE Samples")

In [ ]:
mixed_sample = df.sample(20, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(4, 5, figsize=(12, 10))
axes = axes.flatten()

for i, row in mixed_sample.iterrows():
    img = Image.open(row["filepath"]).convert("RGB")
    axes[i].imshow(img)
    axes[i].set_title(f'#{i}')
    axes[i].axis("off")

plt.suptitle("Blind Mixed Samples", fontsize=14)
plt.tight_layout()
plt.show()

mixed_sample[["filepath", "split", "label"]]

관찰 메모:
- FAKE에서 배경 텍스처가 더 부드럽거나 뭉개지는가?
- 물체 경계가 비정상적으로 번지거나 깨지는가?
- 특정 색감/조명 경향이 있는가?

## 6. Basic Distribution Analysis

In [ ]:
count_table = df.groupby(["split", "label"]).size().reset_index(name="count")
count_table

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for split in ["train", "test"]:
    sub = count_table[count_table["split"] == split]
    ax.bar(
        [f"{split}_{x}" for x in sub["label"]],
        sub["count"]
    )

ax.set_title("Image Count by Split and Label")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
features_to_plot = [
    "filesize_kb", "brightness", "contrast",
    "entropy", "lap_var", "edge_density"
]

fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(8, 4 * len(features_to_plot)))

for ax, feat in zip(axes, features_to_plot):
    real_vals = df[df["label"] == "REAL"][feat].dropna()
    fake_vals = df[df["label"] == "FAKE"][feat].dropna()
    
    ax.hist(real_vals, bins=50, alpha=0.6, label="REAL")
    ax.hist(fake_vals, bins=50, alpha=0.6, label="FAKE")
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
summary_stats = df.groupby("label")[[
    "filesize_kb", "brightness", "contrast",
    "entropy", "lap_var", "edge_density",
    "r_mean", "g_mean", "b_mean"
]].agg(["mean", "std", "median"]).round(4)

summary_stats

여기서 보고 싶은 것:
- REAL / FAKE의 밝기 차이
- contrast / entropy 차이
- sharpness나 edge density 차이
- RGB 평균 차이

## 7. RGB / 픽셀 수준 비교

In [ ]:
rgb_features = ["r_mean", "g_mean", "b_mean", "r_std", "g_std", "b_std"]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for ax, feat in zip(axes, rgb_features):
    real_vals = df[df["label"] == "REAL"][feat].dropna()
    fake_vals = df[df["label"] == "FAKE"][feat].dropna()
    
    ax.hist(real_vals, bins=40, alpha=0.6, label="REAL")
    ax.hist(fake_vals, bins=40, alpha=0.6, label="FAKE")
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
sample_real = df[df["label"] == "REAL"].sample(5000, random_state=SEED)
sample_fake = df[df["label"] == "FAKE"].sample(5000, random_state=SEED)

def collect_pixels(sample_df):
    pixels = []
    for path in tqdm(sample_df["filepath"], total=len(sample_df)):
        img = Image.open(path).convert("RGB")
        arr = np.array(img).reshape(-1, 3)
        pixels.append(arr)
    return np.concatenate(pixels, axis=0)

real_pixels = collect_pixels(sample_real)
fake_pixels = collect_pixels(sample_fake)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, color_name in enumerate(["R", "G", "B"]):
    axes[i].hist(real_pixels[:, i], bins=50, alpha=0.6, label="REAL")
    axes[i].hist(fake_pixels[:, i], bins=50, alpha=0.6, label="FAKE")
    axes[i].set_title(f"{color_name} channel distribution")
    axes[i].legend()

plt.tight_layout()
plt.show()

## 8. Frequency-Domain Analysis

공간 도메인에서는 잘 안 보이는 차이가 주파수 영역에서 나타나는지 확인한다.

In [ ]:
fft_records = []
fft_maps_real = []
fft_maps_fake = []

sample_for_fft = df.groupby(["split", "label"], group_keys=False).apply(
    lambda x: x.sample(min(2000, len(x)), random_state=SEED)
).reset_index(drop=True)

for _, row in tqdm(sample_for_fft.iterrows(), total=len(sample_for_fft)):
    img = Image.open(row["filepath"]).convert("RGB")
    f = compute_fft_features(img)
    
    fft_records.append({
        "filepath": row["filepath"],
        "label": row["label"],
        "split": row["split"],
        "low_freq_energy": f["low_freq_energy"],
        "high_freq_energy": f["high_freq_energy"],
        "high_low_ratio": f["high_low_ratio"],
        "radial_profile": f["radial_profile"]
    })
    
    if row["label"] == "REAL":
        fft_maps_real.append(f["fft_magnitude"])
    else:
        fft_maps_fake.append(f["fft_magnitude"])

fft_df = pd.DataFrame(fft_records)

In [ ]:
fft_df.groupby("label")[["low_freq_energy", "high_freq_energy", "high_low_ratio"]].agg(["mean", "std"]).round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, ["low_freq_energy", "high_freq_energy", "high_low_ratio"]):
    real_vals = fft_df[fft_df["label"] == "REAL"][feat]
    fake_vals = fft_df[fft_df["label"] == "FAKE"][feat]
    
    ax.hist(real_vals, bins=40, alpha=0.6, label="REAL")
    ax.hist(fake_vals, bins=40, alpha=0.6, label="FAKE")
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
mean_fft_real = np.mean(np.stack(fft_maps_real), axis=0)
mean_fft_fake = np.mean(np.stack(fft_maps_fake), axis=0)
fft_diff = mean_fft_fake - mean_fft_real

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(mean_fft_real, cmap="gray")
axes[0].set_title("Mean FFT - REAL")
axes[0].axis("off")

axes[1].imshow(mean_fft_fake, cmap="gray")
axes[1].set_title("Mean FFT - FAKE")
axes[1].axis("off")

axes[2].imshow(fft_diff, cmap="bwr")
axes[2].set_title("FFT Difference (FAKE - REAL)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
real_profiles = np.stack(fft_df[fft_df["label"] == "REAL"]["radial_profile"].values)
fake_profiles = np.stack(fft_df[fft_df["label"] == "FAKE"]["radial_profile"].values)

min_len = min(real_profiles.shape[1], fake_profiles.shape[1])
real_mean_profile = real_profiles[:, :min_len].mean(axis=0)
fake_mean_profile = fake_profiles[:, :min_len].mean(axis=0)

plt.figure(figsize=(8, 5))
plt.plot(real_mean_profile, label="REAL")
plt.plot(fake_mean_profile, label="FAKE")
plt.title("Mean Radial Frequency Profile")
plt.xlabel("Radius")
plt.ylabel("Log Magnitude")
plt.legend()
plt.show()

해석 포인트:
- FAKE가 고주파 성분을 더 많이 가지는가, 혹은 덜 가지는가?
- 특정 주파수 대역에서 차이가 집중되는가?
- 이 차이가 frequency branch 설계의 근거가 될 수 있는가?

## 9. Embedding-Space Exploration

간단한 특징 벡터로 차원을 축소해 REAL / FAKE가 어느 정도 분리되는지 살펴본다.

In [ ]:
from torchvision import models, transforms
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
embed_df = df.groupby(["split", "label"], group_keys=False).apply(
    lambda x: x.sample(min(1000, len(x)), random_state=SEED)
).reset_index(drop=True)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform
        
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, path

dataset = ImagePathDataset(embed_df["filepath"].tolist(), transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model = model.to(device).eval()

In [ ]:
embeddings = []
paths = []

with torch.no_grad():
    for imgs, batch_paths in tqdm(loader):
        imgs = imgs.to(device)
        feats = model(imgs)
        embeddings.append(feats.cpu().numpy())
        paths.extend(batch_paths)

embeddings = np.concatenate(embeddings, axis=0)
embeddings.shape

In [ ]:
embed_vis_df = embed_df.copy().reset_index(drop=True)

pca = PCA(n_components=2, random_state=SEED)
pca_2d = pca.fit_transform(embeddings)

embed_vis_df["pca1"] = pca_2d[:, 0]
embed_vis_df["pca2"] = pca_2d[:, 1]

In [ ]:
plt.figure(figsize=(8, 6))

for label in ["REAL", "FAKE"]:
    sub = embed_vis_df[embed_vis_df["label"] == label]
    plt.scatter(sub["pca1"], sub["pca2"], s=10, alpha=0.5, label=label)

plt.title("PCA of Image Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca")
tsne_2d = tsne.fit_transform(embeddings[:2000])

tsne_df = embed_vis_df.iloc[:2000].copy()
tsne_df["x"] = tsne_2d[:, 0]
tsne_df["y"] = tsne_2d[:, 1]

plt.figure(figsize=(8, 6))
for label in ["REAL", "FAKE"]:
    sub = tsne_df[tsne_df["label"] == label]
    plt.scatter(sub["x"], sub["y"], s=10, alpha=0.5, label=label)

plt.title("t-SNE of Image Embeddings")
plt.legend()
plt.show()

해석 포인트:
- REAL / FAKE가 완전히 섞이는가?
- 일부 군집이 한쪽 라벨에 치우치는가?
- outlier 샘플은 어떤 특징을 가지는가?

## 10. Duplicate / Leakage Check

In [ ]:
import hashlib

def file_md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

dup_sample_df = df.sample(min(10000, len(df)), random_state=SEED).copy()
dup_sample_df["md5"] = dup_sample_df["filepath"].progress_apply(file_md5)

In [ ]:
exact_dups = dup_sample_df[dup_sample_df.duplicated("md5", keep=False)].sort_values("md5")
exact_dups.head(20)

In [ ]:
train_md5 = dup_sample_df[dup_sample_df["split"] == "train"][["filepath", "md5"]]
test_md5 = dup_sample_df[dup_sample_df["split"] == "test"][["filepath", "md5"]]

cross_dup = train_md5.merge(test_md5, on="md5", suffixes=("_train", "_test"))
cross_dup.head()

In [ ]:
from PIL import Image
import imagehash

phash_df = df.sample(min(5000, len(df)), random_state=SEED).copy()
phash_df["phash"] = phash_df["filepath"].progress_apply(
    lambda x: str(imagehash.phash(Image.open(x).convert("RGB")))
)

near_dups = phash_df[phash_df.duplicated("phash", keep=False)].sort_values("phash")
near_dups.head(20)

여기서는
- exact duplicate
- train-test 중복
- perceptual hash 기준 유사중복
가능성을 간단히 점검한다.

## 11. 어려운 샘플 / 이상치 저장

In [ ]:
# 예시: entropy 낮은 샘플, high_low_ratio 높은 샘플 등
hard_candidates = df.copy()

hard_candidates["low_entropy_rank"] = hard_candidates["entropy"].rank(method="first")
hard_candidates["high_lap_rank"] = hard_candidates["lap_var"].rank(method="first", ascending=False)

hard_candidates = hard_candidates.sort_values(["entropy", "lap_var"], ascending=[True, False])
hard_candidates[["filepath", "split", "label", "entropy", "lap_var"]].head(50)

In [ ]:
hard_candidates[[
    "filepath", "split", "label",
    "brightness", "contrast", "entropy", "lap_var", "edge_density"
]].to_csv("cifake_hard_candidates.csv", index=False)

## 12. Findings for Modeling

### 요약
- 데이터 구조는 train/test, REAL/FAKE 이진 분류 형태로 정리되어 있다.
- 이미지 해상도와 채널 수가 대부분 일관적이다.
- REAL / FAKE 간 밝기, 엔트로피, sharpness, edge density 등에서 차이가 존재할 수 있다.
- 주파수 분석에서 특정 대역 차이가 관찰되면 frequency branch 도입의 근거가 된다.
- 임베딩 공간에서 일부 군집 분리가 보이면 baseline CNN만으로도 성능이 나올 수 있지만,
  동시에 artifact 중심 학습 가능성도 고려해야 한다.
- 따라서 이후 실험은
  1) spatial baseline,
  2) frequency-only baseline,
  3) spatial-frequency fusion
  순서로 진행하는 것이 타당하다.